<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">

# **Visión por Computadora 3D**
# **Tarea 3, VGGT con un video propio**

Lo mismo que hicimos con KITTI, pero con otro video y la posibilidad de subir un video grabado por nosotros con el celular.

De un video sacamos algunas fotos y se las damos a VGGT, que en una sola pasada nos devuelve:

- la posición de la cámara en cada foto
- la profundidad de cada píxel
- los puntos 3D de la escena

**Considerar lo siguiente al subir un video propio** (10 a 30 segundos alcanzan):

* caminar despacio alrededor de un objeto o por una habitación, sin girar bruscamente
* que la escena esté quieta (sin gente ni mascotas moviéndose)
* mejor con textura: libros, plantas, muebles. Las paredes lisas y los vidrios cuestan
* grabar en horizontal: si el video es vertical, VGGT recorta arriba y abajo

---
**Objetivo:** reconstruir en 3D una escena a partir de un video casero con VGGT, y analizar cómo influyen la cantidad de fotos, su orden y los objetos en movimiento.

**Descripción:** el notebook extrae fotos repartidas a lo largo de un video y se las da juntas a VGGT, que en una sola pasada estima la posición de la cámara, la profundidad de cada píxel y la nube de puntos 3D de la escena.

---
####**Instrucciones:**
Utilice este notebook como base para realizar el TP. Ejecútelo primero sin cambios y registre los resultados como línea de base. En cada ejercicio modifique solo lo indicado. Restablezca el valor original antes del siguiente ejercicio.

**Parte A) Los siguientes 2 ejercicios son para el video de la pirámide**

1. Cantidad de fotos: reemplace N_FOTOS = 30 por N_FOTOS = 10, luego por 25 y luego por 40.

¿Mejora la nube? ¿Cuándo empieza a fallar?

2. El orden no importa: VGGT mira todas las fotos juntas. Mezcle FOTOS al azar antes de la sección 3 con np.random.shuffle(FOTOS) y compare la nube. ¿Pasaría lo mismo con la odometría visual?

**Parte B) Video propio**

Video propio: suba un video propio y responda las preguntas 1 y 2 para ese video y grabe una escena con alguien caminando. ¿Qué pasa con esa persona en la nube?


**Entregue el notebook descargado en este link https://forms.gle/t1jjETaro6mfrdCY9 con las celdas ejecutadas, de modo que se vean las salidas, los gráficos y sus respuestas.**

---

---
## **0.- Preparación**

In [ ]:
# Descargamos el código de VGGT y las librerías que necesita
!git clone -q https://github.com/facebookresearch/vggt.git vggt_repo
!pip install -q einops safetensors huggingface_hub

In [ ]:
# Importamos las librerías que vamos a usar
import sys
sys.path.append("vggt_repo")

import os
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri

## **1.- El video**

Para subir un video propio puedes descomentar el bloque de abajo y saltarte a la ejecución de la celda que empieza con "#Abrimos el video..."

In [ ]:
'''
VIDEO = ""   # ruta al video, por ejemplo "/content/mi_video.mp4" (vacío = subirlo)

if not VIDEO:
    from google.colab import files
    print("elige tu video")
    VIDEO = list(files.upload())[0]
'''

In [ ]:
!wget -q https://huggingface.co/spaces/facebook/vggt/resolve/main/examples/videos/pyramid.mp4
VIDEO = "pyramid.mp4"

In [ ]:
# Abrimos el video y miramos cuántos cuadros tiene y a cuántos cuadros por segundo
video = cv2.VideoCapture(VIDEO)
TOTAL = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
FPS = video.get(cv2.CAP_PROP_FPS)
print(f"{TOTAL} cuadros a {FPS:.0f} por segundo = {TOTAL / FPS:.1f} segundos")

## **2.- Las fotos**

Un video tiene cientos de cuadros, pero VGGT gasta memoria según cuántas fotos le damos: en una T4 entran unas **30 a 40**. Además, dos cuadros seguidos son casi iguales, así que no aportan nada nuevo.

Tomamos `N_FOTOS` cuadros **repartidos parejo** en todo el video

In [ ]:
N_FOTOS = 30   # bajarlo si te quedas sin memoria, subirlo si tienes una GPU grande

# los cuadros que vamos a usar, repartidos a lo largo del video
CUADROS = np.linspace(0, TOTAL - 1, N_FOTOS).round().astype(int)

# leemos esos cuadros y los guardamos como png
os.makedirs("fotos", exist_ok=True)
FOTOS = []
for k in CUADROS:
    video.set(cv2.CAP_PROP_POS_FRAMES, k)   # saltamos al cuadro k
    ok, cuadro = video.read()
    if ok:
        FOTOS.append(f"fotos/{k:05d}.png")
        cv2.imwrite(FOTOS[-1], cuadro)
video.release()
print("Fotos que le daremos a VGGT:", len(FOTOS))

# miramos tres de ellas: al inicio, al medio y al final
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for a, f in zip(ax, [FOTOS[0], FOTOS[len(FOTOS) // 2], FOTOS[-1]]):
    a.imshow(plt.imread(f))
    a.set_title(f"cuadro {int(f[6:11])} (segundo {int(f[6:11]) / FPS:.1f})")
    a.axis("off")
plt.show()

## **3.- Pasamos las fotos por VGGT**

Igual que con KITTI: **todas las fotos juntas**, una sola pasada

In [ ]:
# Usamos la GPU si hay una (en CPU también corre, pero mucho más lento)
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("Usamos:", dispositivo)

# Descargamos el modelo preentrenado (unos 5 GB, tarda un poco la primera vez)
modelo = VGGT.from_pretrained("facebook/VGGT-1B").to(dispositivo).eval()

# VGGT achica las fotos a 518 píxeles de ancho
imagenes = load_and_preprocess_images(FOTOS).to(dispositivo)
print("Fotos que recibe VGGT:", tuple(imagenes.shape))   # (fotos, 3, alto, ancho)

In [ ]:
# Elegimos la precisión: bfloat16 en GPUs nuevas, float16 en las más viejas (como la T4)
if dispositivo == "cuda" and torch.cuda.get_device_capability()[0] >= 8:
    tipo = torch.bfloat16
else:
    tipo = torch.float16

# Una sola pasada por el modelo
with torch.no_grad(), torch.autocast("cuda", dtype=tipo, enabled=(dispositivo == "cuda")):
    pred = modelo(imagenes)

# Pasamos todo a numpy, igual que en la clase
extr, intr = pose_encoding_to_extri_intri(pred["pose_enc"], imagenes.shape[-2:])
extr = extr[0].float().cpu().numpy()                              # [R | t] de cada foto
intr = intr[0].float().cpu().numpy()                              # K de cada foto
profundidad = pred["depth"][0, ..., 0].float().cpu().numpy()      # (fotos, alto, ancho)
confianza = pred["depth_conf"][0].float().cpu().numpy()
puntos = pred["world_points"][0].float().cpu().numpy()            # (fotos, alto, ancho, 3)
conf_puntos = pred["world_points_conf"][0].float().cpu().numpy()
colores = imagenes.permute(0, 2, 3, 1).cpu().numpy()              # (fotos, alto, ancho, 3)
H, W = profundidad.shape[1:]

# liberamos la GPU
del modelo, pred
torch.cuda.empty_cache()
print("Listo:", len(FOTOS), "fotos de", W, "x", H, "píxeles")

## **4.- Profundidad de cada píxel**

Miramos cuatro fotos del video con su profundidad y su confianza

In [ ]:
elegidas = np.linspace(0, len(FOTOS) - 1, 4).round().astype(int)

fig, ax = plt.subplots(3, 4, figsize=(20, 10))
for col, i in enumerate(elegidas):
    ax[0, col].imshow(colores[i]); ax[0, col].set_title(f"foto {i}")
    ax[1, col].imshow(profundidad[i], cmap="turbo_r")
    ax[2, col].imshow(confianza[i], cmap="viridis")
ax[1, 0].set_title("profundidad (rojo = cerca, azul = lejos)", loc="left")
ax[2, 0].set_title("confianza (oscuro = el modelo duda)", loc="left")
for a in ax.ravel():
    a.axis("off")
plt.show()

¿Dónde duda el modelo? Fíjate en las ventanas, los reflejos, el cielo, los bordes de los objetos y las zonas movidas o borrosas

## **5.- La cámara del celular**

VGGT también estima la **K** de cada foto, sin calibrar nada. Todas las fotos son de la misma camara, así que la focal debería salir **parecida** en todas

In [ ]:
focal = intr[:, 0, 0]
# ángulo de visión horizontal: cuánto "abarca" la cámara
fov = np.degrees(2 * np.arctan(W / (2 * focal)))

print(f"focal: {focal.mean():.0f} px (en fotos de {W} px de ancho) | varía {focal.std() / focal.mean():.0%} entre fotos")
print(f"ángulo de visión horizontal: {fov.mean():.0f}°")
print("La cámara principal de un celular suele tener entre 65° y 80°")

## **6.- ¿Por dónde anduvo la cámara?**

Igual que en la clase: el centro de cada cámara es

$$C = -R^T \, t$$

y el mundo es la **primera foto**. Ahora no sabemos ninguna medida real (no hay altura de cámara conocida), así que todo queda en unidades de VGGT

In [ ]:
# Calculamos el centro de la cámara en cada foto
centros = np.array([-E[:, :3].T @ E[:, 3] for E in extr])
# y hacia dónde mira: la tercera fila de R es el eje z de la cámara visto desde el mundo
mira = np.array([E[2, :3] for E in extr])

# Vista desde arriba: adelante (z) e izquierda (-x), como en la clase
plt.figure(figsize=(10, 7))
plt.plot(centros[:, 2], -centros[:, 0], "tab:orange", lw=2, marker="o")
plt.quiver(centros[:, 2], -centros[:, 0], mira[:, 2], -mira[:, 0], color="gray", width=0.003)
plt.plot(centros[0, 2], -centros[0, 0], "go", ms=12, label="inicio")
plt.plot(centros[-1, 2], -centros[-1, 0], "ro", ms=12, label="fin")
plt.axis("equal"); plt.legend()
plt.xlabel("adelante (unidades VGGT)"); plt.ylabel("izquierda (unidades VGGT)")
plt.title("camino de la cámara (las flechas indican hacia dónde mira)")
plt.show()

**Para pensar:** si conoces alguna medida de la escena (el alto de una mesa, el ancho de una puerta), puedes medirla en la nube 3D y convertir todo a metros, como hicimos con la altura de la cámara en KITTI

## **7.- La escena en 3D**

Juntamos los puntos de todas las fotos con su color. `CONF` decide qué porcentaje de los puntos menos confiables descartamos: probar con 30, 50 y 80

In [ ]:
CONF = 50     # descartamos el CONF % de puntos menos confiables

# Juntamos los puntos de todas las fotos
Q = puntos.reshape(-1, 3)
C = colores.reshape(-1, 3)
c = conf_puntos.reshape(-1)

# Nos quedamos con los más confiables y tomamos 60 000 al azar para que el gráfico no sea lento
buenos = np.flatnonzero(c > np.percentile(c, CONF))
idx = np.random.choice(buenos, min(60000, len(buenos)), replace=False)
Q, C = Q[idx], (C[idx] * 255).astype(int)

# Ordenamos los ejes como en la clase: adelante, izquierda, arriba
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=Q[:, 2], y=-Q[:, 0], z=-Q[:, 1], mode="markers", name="puntos",
                           marker=dict(size=1.2, color=[f"rgb({r},{g},{b})" for r, g, b in C])))
fig.add_trace(go.Scatter3d(x=centros[:, 2], y=-centros[:, 0], z=-centros[:, 1],
                           mode="lines+markers", name="cámara", line=dict(color="red", width=5), marker=dict(size=3)))
fig.update_layout(height=650, title="Mi escena reconstruida por VGGT a partir de un video",
                  scene=dict(aspectmode="data"))
fig.show()

# **Respuestas**
## **Parte A) los siguientes 2 ejercicios son para el video de la piramide**

#### **1.- Cambiar N_FOTOS a 10, 25 y a 40. ¿Mejora la nube? ¿Cuándo empieza a fallar?**

In [ ]:
# Respuesta:

#### **2.- El orden no importa: VGGT mira todas las fotos juntas. Mezclar FOTOS al azar antes de la sección 3 (np.random.shuffle(FOTOS)) y comparar la nube. ¿Pasaría lo mismo con la odometría visual?**

In [ ]:
# Respuesta:

## **Parte B)**

#### **Sube un video propio, responde las preguntas 1 y 2 para ese video y graba una escena con alguien caminando. ¿Qué pasa con esa persona en la nube?**

In [ ]:
# Respuesta: